# Evaluate the testing pipeline

Configure the input paths in the setup cell, then run the cells in order. See the project README for input schemas and model setup.


In [ ]:
from pathlib import Path
import sys

# Locate the project when Jupyter starts in a notebook subfolder.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "smmnlp" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter inside SMMnlp_v2 or set its working directory there.")
sys.path.insert(0, str(project_root))


In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

from smmnlp.evaluation import calculate_binary_metrics, print_metrics
from smmnlp.paths import ProjectPaths
from smmnlp.parsing import expand_partial_date, parse_diagnosis_label
from smmnlp.preprocessing import add_parsed_labels

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
paths = ProjectPaths.from_environment()
# Override paths here, or set SMMNLP_DATA_ROOT and SMMNLP_MODEL_PATH.

## Patient-level hx sMM Date extraction 1. Find patients LLM labeled as positive for SMM (n=200, n=83) 2. Identify the first positive SMM encournter 3. Identify the first positive SMM encournter LLM labeled as hx (n=72, n=34) 4. Extract the date of dx 5. Standardize the date of dx 6. Patients LLM labeled as negative for SMM -> Administrative date Patients LLM labeled as positive for SMM + LLM labeled new for first encounter -> Administrative date


## Read in true labels


In [ ]:
readfiletrue = paths.data("Update/True/SMMNLP_training_true_phase1.xlsx")
true_training = pd.read_excel(readfiletrue)

In [ ]:
readfiletrue = paths.data("Update/True/SMMNLP_testing_true_phase1.xlsx")
true_testing = pd.read_excel(readfiletrue)

In [ ]:
train_ids = true_training["PatientSSN"].unique()
test_ids = true_testing["PatientSSN"].unique()

In [ ]:
len(test_ids)

## Reference Standard


In [ ]:
readfile_name = paths.data("Update/True/EndToEnd_True_testing.xlsx")
true_all = pd.read_excel(readfile_name)[["PatientSSN", "True_SMMDx_Binary", "True_SMMDx_Date"]]

In [ ]:
len(true_all)

In [ ]:
true_all.head(5)

## LLM Results


In [ ]:
readfileresults = paths.data("Random500/Results/testing_fewshot_binary.xlsx")
results_s1 = pd.read_excel(readfileresults)[
    ["PatientSSN", "NoteDate", "NoteText", "Pt_index", "Note_index", "SMMDx_LLM_Binary"]
]

In [ ]:
results_s1.head(5)

In [ ]:
readfileresults = paths.data("Second237/Results/phase1_fewshot_binary.xlsx")
results_s2 = pd.read_excel(readfileresults)[
    ["PatientSSN", "NoteDate", "NoteText", "Pt_index", "Note_index", "SMMDx_LLM_Binary"]
]

In [ ]:
results_s2.head(5)

In [ ]:
combined_results = pd.concat([results_s1, results_s2], ignore_index=True)

In [ ]:
len(combined_results)

In [ ]:
results_all = combined_results[combined_results["PatientSSN"].isin(test_ids)]

In [ ]:
len(results_all)

## POST PROCESSING


In [ ]:
llm_columns = ["SMMDx_LLM_Binary"]
results_all = add_parsed_labels(results_all, llm_columns, parse_diagnosis_label)

In [ ]:
smmyes = results_all[results_all["post_SMMDx_LLM_Binary"] == "Yes"]
smmyes = smmyes[["PatientSSN", "post_SMMDx_LLM_Binary", "NoteDate"]]

In [ ]:
len(smmyes)

In [ ]:
smmyes.head(5)

In [ ]:
smmyes["NoteDate2"] = pd.to_datetime(smmyes["NoteDate"], format="%d%b%Y")

In [ ]:
smmyes_cleaned = smmyes.loc[smmyes.groupby("PatientSSN")["NoteDate2"].idxmin()]

In [ ]:
len(smmyes_cleaned)

In [ ]:
smmyes_cleaned.head(5)

In [ ]:
datedata = paths.data("Update/ExtraLLMRun/EndToEnd_testing_dxdate_human.xlsx")
datedata_all = pd.read_excel(datedata)[
    ["PatientSSN", "DxDate_LLM", "UncertainFlag_LLM", "DxDate_Human"]
]

In [ ]:
datedata_all.head()

In [ ]:
len(datedata_all)

## POST PROCESSING


In [ ]:
datedata_all["DxDate_Reviewed"] = np.where(
    datedata_all["UncertainFlag_LLM"] == True,
    datedata_all["DxDate_Human"],
    datedata_all["DxDate_LLM"],
)

In [ ]:
datedata_all.head(15)

In [ ]:
date_diff = datedata_all[datedata_all["DxDate_Reviewed"] != "unknown"]

In [ ]:
len(date_diff)

In [ ]:
date_diff["DxDate_LLM2"] = date_diff["DxDate_Reviewed"].apply(expand_partial_date)

In [ ]:
date_diff.head(5)

In [ ]:
date_diff["DxDate_LLM_Clean"] = pd.to_datetime(date_diff["DxDate_LLM2"], format="%m/%d/%Y")

In [ ]:
smmhx = datedata_all.merge(
    date_diff[["PatientSSN", "DxDate_LLM_Clean"]], on=["PatientSSN"], how="left"
)

In [ ]:
len(smmhx)

In [ ]:
smmhx.head(10)

In [ ]:
len(smmyes_cleaned)

In [ ]:
true_all.head(5)

In [ ]:
smmyes_cleaned.head(5)

In [ ]:
smmfull_true1 = true_all.merge(smmyes_cleaned, on=["PatientSSN"], how="left")

In [ ]:
smmfull_true2 = pd.merge(smmfull_true1, smmhx, on=["PatientSSN"], how="left")

In [ ]:
len(smmfull_true2)

In [ ]:
smmfull_true2.head(5)

In [ ]:
freq_table = smmfull_true2["post_SMMDx_LLM_Binary"].value_counts()
print(freq_table)

In [ ]:
smmfull_true2["LLM_SMMDx_Binary"] = smmfull_true2["post_SMMDx_LLM_Binary"].fillna(
    "No"
)

In [ ]:
freq_table = smmfull_true2["LLM_SMMDx_Binary"].value_counts()
print(freq_table)

In [ ]:
smmfull_true2.head(5)

In [ ]:
smmfull_true2["LLM_SMMDx_Date"] = np.where(
    smmfull_true2["DxDate_LLM_Clean"].notna(),
    smmfull_true2["DxDate_LLM_Clean"],
    np.where(smmfull_true2["DxDate_Reviewed"] == "unknown", pd.NaT, smmfull_true2["NoteDate2"]),
)

In [ ]:
final = smmfull_true2[
    ["PatientSSN", "True_SMMDx_Binary", "True_SMMDx_Date", "LLM_SMMDx_Binary", "LLM_SMMDx_Date"]
]

In [ ]:
final.head(5)

In [ ]:
len(final)

## End-to-End Performance


In [ ]:
final["y_true"] = final["True_SMMDx_Binary"].astype(int)
final["y_pred"] = final["LLM_SMMDx_Binary"].str.lower().map({"yes": 1, "no": 0})

In [ ]:
y_true = final["y_true"]
y_pred = final["y_pred"]
metric_result = calculate_binary_metrics(y_true, y_pred)

In [ ]:
cm_display = ConfusionMatrixDisplay(
    confusion_matrix=metric_result.cm, display_labels=["Non-SMM", "SMM"]
)
cm_display.plot()
plt.show()

In [ ]:
print_metrics(metric_result)

In [ ]:
unknowndate = final[(final["True_SMMDx_Binary"] == True) & (final["LLM_SMMDx_Binary"] == "Yes")]

In [ ]:
len(unknowndate)

In [ ]:
unknowndate.head()

In [ ]:
conditions = [
    unknowndate["LLM_SMMDx_Date"].notna() & unknowndate["True_SMMDx_Date"].notna(),
    ~unknowndate["LLM_SMMDx_Date"].notna() & ~unknowndate["True_SMMDx_Date"].notna(),
    ~unknowndate["LLM_SMMDx_Date"].notna() & unknowndate["True_SMMDx_Date"].notna(),
    unknowndate["LLM_SMMDx_Date"].notna() & ~unknowndate["True_SMMDx_Date"].notna(),
]
choices = ["TP", "TN", "FN", "FP"]
unknowndate["Outcome3_result1"] = np.select(conditions, choices, default="Unknown")

In [ ]:
print(unknowndate["Outcome3_result1"].value_counts())

In [ ]:
unknowndate["y_true"] = np.select([unknowndate["True_SMMDx_Date"].notna()], [1], default=0)
unknowndate["y_pred"] = np.select([unknowndate["LLM_SMMDx_Date"].notna()], [1], default=0)

In [ ]:
y_true = unknowndate["y_true"]
y_pred = unknowndate["y_pred"]
metric_result = calculate_binary_metrics(y_true, y_pred)

In [ ]:
cm_display = ConfusionMatrixDisplay(
    confusion_matrix=metric_result.cm,
    display_labels=["SMM dx date not present", "SMM dx date present"],
)
cm_display.plot()
plt.show()

In [ ]:
print_metrics(metric_result)

In [ ]:
date_diff = final[(final["True_SMMDx_Binary"] == True) & (final["LLM_SMMDx_Binary"] == "Yes")]

In [ ]:
len(date_diff)

In [ ]:
date_diff["date_diff"] = date_diff["LLM_SMMDx_Date"] - date_diff["True_SMMDx_Date"]

In [ ]:
date_diff_days = date_diff["date_diff"].dt.days
plt.figure(figsize=(8, 5))
n, bins, patches = plt.hist(date_diff_days.dropna(), bins=30, color="orange", edgecolor="black")
plt.xlabel("LLM-extracted - Reference standard")
plt.ylabel("Frequency")
plt.title("Histogram of date difference (days)")
plt.show()

## TO DELETE date_diff2 = date_diff[ (date_diff['date_diff'] > pd.Timedelta(days=-500)) & (date_diff['date_diff'] < pd.Timedelta(days=500)) ].copy()


In [ ]:
date_diff_days

In [ ]:
x = date_diff["date_diff"].dt.days.dropna().reset_index(drop=True)
y = np.random.uniform(0, 1, len(x))
fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(x, y, s=320, facecolor="#ED8936", edgecolor="#8B4513", linewidth=1.8)
ax.set_xlabel("Differences in Days - End-to-End", fontsize=26, color="#8B4513", labelpad=15)
ax.set_yticks([])
ax.set_facecolor("#f3f3f3")
ax.grid(axis="x", color="#d0d0d0", linewidth=1.2)
ax.set_axisbelow(True)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis="x", labelsize=20, colors="#8B4513", length=0)
plt.tight_layout()
plt.show()

In [ ]:
savefile_name = paths.data("Update/EndToEnd_testing_dataset_datediff_human.xlsx")
date_diff_days.to_excel(savefile_name, engine="xlsxwriter", sheet_name="Sheet1", index=False)